# OfficeQA Retrieval Runner

This notebook is a thin Colab wrapper around the Python package in this repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/hungngo2/cs445-final-project-retrieval.git officeqa-retrieval
%cd /content/officeqa-retrieval
!python -m pip install -e '.[ml,dev]'

In [ ]:
QUESTIONS_CSV = '/content/drive/MyDrive/officeqa/officeqa_pro.csv'
PDF_DIR = '/content/drive/MyDrive/officeqa/pdfs'
ARTIFACT_DIR = '/content/drive/MyDrive/officeqa/artifacts'
RESULTS_DIR = '/content/drive/MyDrive/officeqa/results'

!python scripts/prepare_data.py --questions-csv "$QUESTIONS_CSV" --pdf-dir "$PDF_DIR" --manifest-out "$ARTIFACT_DIR/page_manifest.jsonl" --sanity-out "$ARTIFACT_DIR/sanity_questions.json"
!python scripts/build_page_index.py --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index-out "$ARTIFACT_DIR/page_bm25.pkl"

In [ ]:
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/bm25" --model bm25 --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/clip_full" --model clip --crop-mode full --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/clip_fixed" --model clip --crop-mode fixed_2x2 --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/siglip_full" --model siglip --crop-mode full --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/siglip_fixed" --model siglip --crop-mode fixed_2x2 --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/make_report_tables.py --results-dir "$RESULTS_DIR" --summary-out "$RESULTS_DIR/summary.csv"